In [1]:
import os
os.environ['HADOOP_CONF_DIR'] = '/etc/hadoop/conf'
os.environ['YARN_CONF_DIR'] = '/etc/hadoop/conf'

import findspark
findspark.init()
findspark.find()

'/opt/spark'

In [30]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
                    .master("local") \
                    .appName("Learning DataFrames") \
                    .getOrCreate()
# данные  датафрейма 
data = [('2021-01-06', 3744, 63, 322),
        ('2021-01-04', 2434, 21, 382),
        ('2021-01-04', 2434, 32, 159),
        ('2021-01-05', 3744, 32, 159),
        ('2021-01-06', 4342, 32, 159),
        ('2021-01-05', 4342, 12, 259),
        ('2021-01-06', 5677, 12, 259),
        ('2021-01-04', 5677, 23, 499)
]
# названия атрибутов
columns = ['dt', 'user_id', 'product_id', 'purchase_amount']
# создаём датафрейм
df = spark.createDataFrame(data=data, schema=columns)

## Task 1

In [31]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

window = Window().partitionBy().orderBy(F.asc('purchase_amount'))

df_window = df.withColumn("row_number", F.row_number().over(window))

df_window.select('dt', 'user_id', 'purchase_amount', 'row_number').show()

26/08/19 12:02:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/19 12:02:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/19 12:02:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
+----------+-------+---------------+----------+
|        dt|user_id|purchase_amount|row_number|
+----------+-------+---------------+----------+
|2021-01-04|   2434|            159|         1|
|2021-01-05|   3744|            159|         2|
|2021-01-06|   4342|            159|         3|
|2021-01-05|   4342|            259|         4|
|2021-01-06|   5677|            259|         5|
|2021-01-06|   3744|            322|         6|
|2021-01-04|   2434|            382|         7|
|2021-01-04|   5677|     

## Task 2

In [9]:
events = spark.read.json("/user/master/data/events/date=2022-05-01")


+--------------------+------------+
|               event|  event_type|
+--------------------+------------+
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
+--------------------+------------+
only showing top 10 rows



In [10]:
events.printSchema()

root
 |-- event: struct (nullable = true)
 |    |-- admins: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- channel_id: long (nullable = true)
 |    |-- datetime: string (nullable = true)
 |    |-- media: struct (nullable = true)
 |    |    |-- media_type: string (nullable = true)
 |    |    |-- src: string (nullable = true)
 |    |-- message: string (nullable = true)
 |    |-- message_channel_to: long (nullable = true)
 |    |-- message_from: long (nullable = true)
 |    |-- message_group: long (nullable = true)
 |    |-- message_id: long (nullable = true)
 |    |-- message_to: long (nullable = true)
 |    |-- message_ts: string (nullable = true)
 |    |-- reaction_from: string (nullable = true)
 |    |-- reaction_type: string (nullable = true)
 |    |-- subscription_channel: long (nullable = true)
 |    |-- tags: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- user: string (nullable = true)
 |-- event_type: s

## Task 3

In [26]:
window = Window() \
        .partitionBy('event.message_from') \
        .orderBy('event.message_ts')


dfWithLag = events.withColumn("lag_7", F.lag("event.message_to", 7).over(window))

dfWithLag.select("event.message_from", "lag_7") \
         .filter(dfWithLag.lag_7.isNotNull()) \
         .orderBy(F.desc('event.message_from')) \
         .show(10, False)


+------------+------+
|message_from|lag_7 |
+------------+------+
|155747      |121581|
|155747      |25843 |
|155747      |24666 |
|155747      |29338 |
|155058      |70776 |
|155058      |37570 |
|155058      |70063 |
|155058      |121908|
|155058      |12334 |
|155058      |70796 |
+------------+------+
only showing top 10 rows



## Task 4

In [43]:
window = Window().partitionBy('user_id')

df_window_agg = df.withColumn("max", F.max("purchase_amount").over(window)) \
                  .withColumn("min", F.min("purchase_amount").over(window))

df_window_agg.select('user_id', 'max', 'min').distinct().show()
                




+-------+---+---+
|user_id|max|min|
+-------+---+---+
|   2434|382|159|
|   3744|322|159|
|   4342|259|159|
|   5677|499|259|
+-------+---+---+

